In [1]:
import json
import os

os.chdir("..")

In [2]:
from cluster_intrep_repo.utils import DOMAIN_PHRASES

In [9]:
# def make_mean_reprs_layer(layer: int):
all_representations = {}

for i in range(1, 16):
    with open(f"multilayer_representations/multilayer_10k/mystery_{i}/mean_reprs_mystery_{i}_10k_multi_layer.json") as f:
        all_representations[f"mystery_{i}"] = json.load(f)

In [10]:
import numpy as np

avg_representations = {k: {} for k in all_representations.keys()}

def process_layer(layer: int):
    representations = {
        k: v[f"{layer}"] for k, v in all_representations.items()
    }
    
    action_reprs = {
        k: {
            kk: np.array(vv) - np.array(v["mean_actions"]) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["actions"].values()
        } for k, v in representations.items()
    }

    predicate_reprs = {
        k: {
            kk: np.array(vv) - np.array(v["mean_predicates"]) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["predicates"].values()
        } for k, v in representations.items()
    }

    # action_reprs = {
    #     k: {
    #         kk: np.array(vv) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["actions"].values()
    #     } for k, v in representations.items()
    # }

    # predicate_reprs = {
    #     k: {
    #         kk: np.array(vv) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["predicates"].values()
    #     } for k, v in representations.items()
    # }
    
    reverse_phrases = {
        k: {kk: {
            vvv: kkk for kkk, vvv in vv.items()
        } for kk, vv in v.items() }
        for k, v in DOMAIN_PHRASES.items()
    }
    
    predicates = list(reverse_phrases["mystery_3"]["predicates"].values())
    actions = list(reverse_phrases["mystery_3"]["actions"].values())
    
    action_reprs = {
        k: {
            reverse_phrases[k]["actions"][kk]: vv for kk, vv in v.items()
        } for k, v in action_reprs.items()
    }

    predicate_reprs = {
        k: {
            reverse_phrases[k]["predicates"][kk]: vv for kk, vv in v.items()
        } for k, v in predicate_reprs.items()
    }
    
    mean_action_reprs = {
        action: np.mean(
            np.stack(
                [action_reprs[m][action] for m in action_reprs]
            ), axis=0
        ) for action in actions
    }
    
    mean_predicate_reprs = {
        pr: np.mean(
            np.stack(
                [predicate_reprs[m][pr] for m in predicate_reprs]
            ), axis=0
        ) for pr in predicates
    }
    
    for i in range(1, 16):
        reprs = {
            "mean_domain": [0] * 5120,
            "mean_actions": [0] * 5120,
            "mean_predicates": [0] * 5120,
            "mean_reprs": {
                DOMAIN_PHRASES[f"mystery_{i}"]["actions"][action]: mean_action_reprs[action].tolist() for action in actions
            }
        }
        
        reprs["mean_reprs"].update({
            DOMAIN_PHRASES[f"mystery_{i}"]["predicates"][predicate]: mean_predicate_reprs[predicate].tolist() for predicate in predicates
        })
            
        avg_representations[f"mystery_{i}"][f"{layer}"] = reprs

In [11]:
from tqdm import tqdm

for i in tqdm(range(50)):
    process_layer(i)

100%|██████████| 50/50 [00:02<00:00, 18.59it/s]


In [12]:
avg_representations["mystery_1"]["2"]["mean_reprs"]["attack"]

[0.050407155354817705,
 0.00020955403645833332,
 0.03159205118815104,
 -0.0710174560546875,
 -0.04394327799479167,
 -0.01063232421875,
 -0.02303492228190104,
 0.008569145202636718,
 0.0244384765625,
 -0.01187744140625,
 0.072369384765625,
 -0.011295572916666666,
 0.019086011250813804,
 0.002520751953125,
 0.10060221354166667,
 0.025144068400065105,
 0.0135528564453125,
 -0.013314310709635417,
 -0.0024489084879557293,
 0.054865519205729164,
 -0.0678118387858073,
 -0.0755340576171875,
 -0.013688151041666667,
 0.050464630126953125,
 0.07293497721354167,
 -0.012142181396484375,
 -0.042711893717447914,
 -0.042052205403645834,
 -0.024532063802083334,
 0.025133260091145835,
 -0.012630208333333334,
 -0.03676808675130208,
 0.013055419921875,
 -0.022239176432291667,
 0.03758570353190104,
 -0.044906997680664064,
 0.021464029947916668,
 0.05590006510416667,
 -0.23249918619791668,
 0.1000030517578125,
 -0.016945902506510416,
 0.0201324462890625,
 -0.058530267079671225,
 -0.011800130208333334,
 -0.0

In [13]:
from pathlib import Path

for i in range(1, 16):
    path = Path(f"multilayer_representations_avg/multilayer_10k/mystery_{i}/mean_reprs_mystery_{i}_multi_layer.json")
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(avg_representations[f"mystery_{i}"], f, indent=4)

: 